In [1]:
%load_ext autoreload
%autoreload 2


In [2]:

from torch.utils.data import DataLoader
from pathlib import Path
import sys
import torch
from hist_colorizer.trainer_hist import TrainerHist
from hist_colorizer.model_hist import UNetHist
from hist_colorizer.dataset import ImagewoofColorizationDataset   # <-- tu dataset

# -----------------------------------------
# AJUSTÁ ESTAS RUTAS
# -----------------------------------------
DATA_DIR = Path("imagewoof2-160")
BATCH_SIZE = 8
EPOCHS = 5

train_ds = ImagewoofColorizationDataset(DATA_DIR, split="train")
val_ds   = ImagewoofColorizationDataset(DATA_DIR, split="val")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)

### Modelo desde cero, sin backbone

In [3]:
model = UNetHist(K=32)

trainer = TrainerHist(model, lr=1e-4)

for epoch in range(1, EPOCHS+1):
    print(f"\n=== Epoch {epoch} ===")
    train_loss = trainer.train_epoch(train_loader)
    val_loss, val_psnr = trainer.validate_epoch(val_loader)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val PSNR:   {val_psnr:.2f} dB")

    # guardar modelo
    torch.save(model.state_dict(), f"hist_model_epoch{epoch}.pt")



=== Epoch 1 ===


Train:  48%|████▊     | 543/1129 [01:52<02:01,  4.81it/s]


KeyboardInterrupt: 

In [4]:
import torch
from torch.utils.data import DataLoader
from hist_colorizer.dataset import ImagewoofColorizationDataset
from hist_colorizer.model_hist import UNetHist
from hist_colorizer.utils_hist import show_colorization

# ------------ CONFIG ------------
CHECKPOINT = "hist_model_epoch5.pt"
DATA_DIR = "imagewoof2-160"
BATCH_SIZE = 4
# --------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

# cargar dataset
ds = ImagewoofColorizationDataset(DATA_DIR, split="val")
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

# cargar modelo
model = UNetHist(K=32).to(device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))

# mostrar 4 imágenes
show_colorization(model, loader, device=device)


FileNotFoundError: [Errno 2] No such file or directory: 'hist_model_epoch5.pt'

In [5]:
def train_model(model, train_loader, val_loader, save_name, criterion="l1"):
    save_path = "pesos_entrenados"
    model_path = Path(save_path) / save_name

    # carpeta donde guardamos las curvas loss vs epoch
    history_dir = Path("loss_vs_epoch")
    history_dir.mkdir(parents=True, exist_ok=True)
    history_path = history_dir / f"{save_name}_history.pt"

    train = False  # Cambia a True para forzar el reentrenamiento

    # Verificar si ya existe un modelo entrenado
    if model_path.exists() and not train:
        print(f"✅Modelo ya entrenado encontrado en '{model_path}'.")
        print("No se vuelve a entrenar para evitar sobreescritura.")

        # (opcional) si ya tenés la history guardada, podés devolverla:
        if history_path.exists():
            print(f"History encontrada en '{history_path}'.")
            history = torch.load(history_path, map_location="cpu")
            return history
        else:
            print("⚠️No se encontró history guardada para este modelo.")
            return None

    else:
        print("🚀No se encontró modelo entrenado, iniciando entrenamiento...")

        # ⬇️ahora trainer devuelve la history
        history = trainer(
            model,
            train_loader,
            val_loader,
            epochs=10,
            save_path=save_path,
            save_name=save_name,
            criterion=criterion,
        )

        print(f"💾Modelo guardado en: {model_path}")

        # ⬇️guardamos la history de forma GENERAL
        torch.save(history, history_path)
        print(f"📈History guardada en: {history_path}")

        return history

## Modelo con bacbone resnet, entrenado con L1 + Histograma

In [ ]:
import os

ruta_proyecto = os.path.abspath("..")
if ruta_proyecto not in sys.path:
    sys.path.append(ruta_proyecto)
from models.unet_resnet import get_model_unet_resnet34
from utils.trainer import train_model

model = get_model_unet_resnet34(pre_entrenado=True, congelar_encoder=False)
save_name = "unet_resnet34_histogram.pt"
train_model(model, train_loader, val_loader, save_name, criterion="histogram")

ModuleNotFoundError: No module named 'models.unet_resnet'